<a href="https://colab.research.google.com/github/Damainx22/RGV-Business-Survival/blob/main/notebooks/01_sba_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [124]:
# ============================================
# Notebook 01: SBA Loan Data Cleaning
# Purpose: Load raw SBA 7(a) loan data,
#          filter to RGV zip codes and FY2018-2022,
#          clean for modeling, save to data/cleaned/
# ============================================

# Mount Google Drive so we can access our raw data files
from google.colab import drive
drive.mount('/content/drive')

# Standard imports
import os
import pandas as pd

# Define folder paths so we don't repeat long strings everywhere
# RAW = where original untouched downloads live
# CLEANED = where filtered/cleaned data gets saved
# MERGED = where the final combined dataset goes (used in notebook 05)
RAW = '/content/drive/MyDrive/rgv_business_survival/data/raw'
CLEANED = '/content/drive/MyDrive/rgv_business_survival/data/cleaned'
MERGED = '/content/drive/MyDrive/rgv_business_survival/data/merged'

# Make sure cleaned folder exists
os.makedirs(CLEANED, exist_ok=True)

# Verify our raw files are accessible
print("Raw files:", sorted(os.listdir(RAW)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Raw files: ['7a_504_foia_data_dictionary.xlsx', 'bds2023_sec.csv', 'bds2023_st_cty.csv', 'foia-7a-fy2010-fy2019-as-of-251231.csv', 'foia-7a-fy2020-present-as-of-251231.csv', 'zbp18detail.txt', 'zbp18detail.zip', 'zbp19detail.txt', 'zbp19detail.zip', 'zbp20detail.txt', 'zbp20detail.zip', 'zbp21detail.txt', 'zbp21detail.zip', 'zbp22detail.txt', 'zbp22detail.zip']


In [125]:
# load both sba loan files to clean and merge
sba_2010 = pd.read_csv(f'{RAW}/foia-7a-fy2010-fy2019-as-of-251231.csv', low_memory=False)
sba_2020 = pd.read_csv(f'{RAW}/foia-7a-fy2020-present-as-of-251231.csv', low_memory=False)

# Stack both files into one DataFrame since they have the same columns
# ignore_index=True resets the row numbers so they don't duplicate
sba_all = pd.concat([sba_2010, sba_2020], ignore_index=True)

# how mnay total lonas loaded and columns
print(f'Total loans loaded: {len(sba_all):,}')
print(f'Columns: {list(sba_all.columns)}')


Total loans loaded: 903,617
Columns: ['asofdate', 'program', 'l2locid', 'borrname', 'borrstreet', 'borrcity', 'borrstate', 'borrzip', 'bankname', 'bankfdicnumber', 'bankncuanumber', 'bankstreet', 'bankcity', 'bankstate', 'bankzip', 'grossapproval', 'sbaguaranteedapproval', 'approvaldate', 'approvalfiscalyear', 'firstdisbursementdate', 'processingmethod', 'subprogram', 'initialinterestrate', 'fixedorvariableinterestind', 'terminmonths', 'naicscode', 'naicsdescription', 'franchisecode', 'franchisename', 'projectcounty', 'projectstate', 'sbadistrictoffice', 'congressionaldistrict', 'businesstype', 'businessage', 'loanstatus', 'paidinfulldate', 'chargeoffdate', 'grosschargeoffamount', 'revolverstatus', 'jobssupported', 'collateralind', 'soldsecmrktind']


In [126]:
#Filter by state (TX) drop rest of rows
sba_all = sba_all[sba_all['borrstate'] == 'TX']

# Filber by string prefix
sba_all['borrzip'] = sba_all['borrzip'].astype(str)
sba_all = sba_all[sba_all['borrzip'].str.startswith('785')]

#check outcome
print(f'Total RGV loans: {len(sba_all)}')
print(sba_all['borrzip'].value_counts().head(10))

Total RGV loans: 1734
borrzip
78501    170
78504    131
78539    110
78572    108
78521    104
78550     96
78596     92
78577     89
78520     87
78526     75
Name: count, dtype: int64


In [127]:
# drop columns until 2019-2022
sba_all = sba_all[sba_all['approvalfiscalyear'].between(2018, 2022)]
print(f'Total RGV loans from 2018-2022: {len(sba_all)}')

Total RGV loans from 2018-2022: 334


In [128]:
# Check year distribution to make sure filter worked right
print(sba_all['approvalfiscalyear'].value_counts().sort_index())
# we can notice here that 2020 during covide not alot of
# businesses took out SBA loans compared to rest of years

approvalfiscalyear
2018    92
2019    83
2020    36
2021    61
2022    62
Name: count, dtype: int64


In [129]:
# check the loan status rows to see what we could use
# for prediction model
print(sba_all['loanstatus'].value_counts())

loanstatus
EXEMPT    148
PIF       120
CANCLD     42
CHGOFF     20
COMMIT      4
Name: count, dtype: int64


In [130]:
# only get PIF(paid in full) and CHGOFF(business defaulted/failed)
sba_all = sba_all[sba_all['loanstatus'].isin(['PIF', 'CHGOFF'])]
print(sba_all['loanstatus'].value_counts())

loanstatus
PIF       120
CHGOFF     20
Name: count, dtype: int64


In [131]:
sba_all['defaulted'] = (sba_all['loanstatus'] == 'CHGOFF')# True/False
sba_all['defaulted'] = sba_all['defaulted'].astype(int) # convert to 1/0

print(sba_all['defaulted'].value_counts())
print(f'Default rate: {sba_all["defaulted"].mean()*100:.1f}%')

defaulted
0    120
1     20
Name: count, dtype: int64
Default rate: 14.3%


In [132]:
# quick visual of loans that have been defaulted in the RGV
print(sba_all[sba_all['loanstatus'] == 'CHGOFF'].head())

          asofdate program   l2locid                   borrname  \
83222   12/31/2025      7A  123499.0     J. AND C. CONTROL INC.   
97097   12/31/2025      7A  509578.0  1929 Bar &amp; Grill Inc.   
102223  12/31/2025      7A  323499.0                  Garan LLC   
161702  12/31/2025      7A   44449.0       RHINO RUSH LOGISTICS   
170633  12/31/2025      7A   72083.0                   UB21 LLC   

                          borrstreet     borrcity borrstate borrzip  \
83222                301 MEXICO BLVD  BROWNSVILLE        TX   78520   
97097                221 Old Stadium  PORT ISABEL        TX   78578   
102223                1006 S 10TH ST      MCALLEN        TX   78501   
161702               201 ALMA STREET      PENITAS        TX   78576   
170633  3400 Expressway 83 Suite 730      McAllen        TX   78501   

                              bankname  bankfdicnumber  ...  \
83222           BayFirst National Bank         34997.0  ...   
97097                       PeopleFund      

In [133]:
# Keep only the 16 columns relevant to the prediction model
# Drops administrative columns like bank info, dates, and IDs we don't need
sba_all = sba_all[['borrname', 'borrcity', 'borrzip', 'grossapproval', 'sbaguaranteedapproval',
                    'approvalfiscalyear', 'initialinterestrate', 'terminmonths', 'naicscode',
                    'naicsdescription', 'businesstype', 'businessage', 'loanstatus', 'defaulted',
                    'jobssupported', 'projectcounty']]

# Verify the result — should show 16 columns, 140 rows, and null counts per column
print(f'Columns: {len(sba_all.columns)}')
print(f'Rows: {len(sba_all)}')
print(sba_all.isnull().sum())

Columns: 16
Rows: 140
borrname                 0
borrcity                 0
borrzip                  0
grossapproval            0
sbaguaranteedapproval    0
approvalfiscalyear       0
initialinterestrate      0
terminmonths             0
naicscode                0
naicsdescription         1
businesstype             0
businessage              3
loanstatus               0
defaulted                0
jobssupported            0
projectcounty            0
dtype: int64


In [135]:
# Fill missing industry descriptions with 'Unknown' to avoid dropping rows
sba_all['naicsdescription'] = sba_all['naicsdescription'].fillna('Unknown')

# Fill missing business age with 'Unknown' — only 3 rows affected
sba_all['businessage'] = sba_all['businessage'].fillna('Unknown')

# Verify no nulls remain
print(sba_all.isnull().sum())

borrname                 0
borrcity                 0
borrzip                  0
grossapproval            0
sbaguaranteedapproval    0
approvalfiscalyear       0
initialinterestrate      0
terminmonths             0
naicscode                0
naicsdescription         0
businesstype             0
businessage              0
loanstatus               0
defaulted                0
jobssupported            0
projectcounty            0
dtype: int64


In [137]:
# Save the cleaned SBA dataset to data/cleaned/ folder
# index=False prevents pandas from writing row numbers as a column
sba_all.to_csv(f'{CLEANED}/sba_rgv_clean.csv', index=False)

# Confirm save was successful by printing shape and first few rows
print(f'Saved: {len(sba_all)} rows, {len(sba_all.columns)} columns')
print(sba_all.head())

Saved: 140 rows, 16 columns
                             borrname     borrcity borrzip  grossapproval  \
11884                     CROSS HOMES   SAN BENITO   78586          25000   
11929     A.S.A GUARDIAN PEST CONTROL        DONNA   78537          20000   
27356  DALET CONSTRUCTION & DECOR LLC  BROWNSVILLE   78526          15000   
42268       RTS CONSTRUCTION SERVICES        ALAMO   78516           5000   
50734       LUCY DRYWALL INCORPORATED      MCALLEN   78501         100000   

       sbaguaranteedapproval  approvalfiscalyear  initialinterestrate  \
11884                12500.0                2018                 9.00   
11929                10000.0                2018                 9.46   
27356                 7500.0                2019                12.00   
42268                 2500.0                2018                10.75   
50734                50000.0                2019                 9.65   

       terminmonths  naicscode                         naicsdescriptio